In [2]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles Python sees in folder:")
print(os.listdir("../data/landmark_raw"))

Current working directory:
C:\Users\User\PycharmProjects\AISS-forest-restoration-project\notebooks

Files Python sees in folder:
['LandMark_IPLC_poly_public_v202509.dbf', 'LandMark_IPLC_poly_public_v202509.prj', 'LandMark_IPLC_poly_public_v202509.sbx', 'LandMark_IPLC_poly_public_v202509.shp', 'LandMark_IPLC_poly_public_v202509.shx', 'LandMark_IPLC_pt_public_v202509.dbf']


In [3]:
import geopandas as gpd

gdf = gpd.read_file("../data/landmark_raw/LandMark_IPLC_poly_public_v202509.shp")

print("Rows:", len(gdf))
print("Columns:", gdf.columns)

Rows: 124616
Columns: Index(['identity', 'name', 'form_rec', 'doc_status', 'stat_date', 'stat_note',
       'country', 'category', 'ethncty_1', 'populatn', 'pop_source',
       'pop_year', 'area_ofcl', 'area_gis', 'scale', 'method', 'data_ctrb',
       'data_src', 'data_src_s', 'data_src_l', 'data_date', 'add_note',
       'more_info', 'layer', 'download', 'iso_code', 'geometry'],
      dtype='str')


In [4]:
gdf["country"].unique()


<StringArray>
[                       'Australia',                           'Brazil',
                         'Cambodia',                           'Canada',
                          'Bolivia',                           'Panama',
                      'New Zealand',                       'Costa Rica',
         'United States of America',              'Greenland (Denmark)',
                         'Malaysia',                     'South Africa',
                         'Botswana',                          'Ecuador',
         'United Kingdom - England',                          'Ireland',
                            'Chile',                           'Zambia',
              'Antigua and Barbuda',                         'Zimbabwe',
                           'Taiwan',                         'Paraguay',
                        'Nicaragua',                           'Guyana',
                         'Cameroon', 'Democratic Republic of the Congo',
                           'Mexico', 

In [5]:
import unicodedata

def remove_accents(text):
    if isinstance(text, str):
        return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    return text

gdf["country"] = gdf["country"].apply(remove_accents)
gdf["area_km2"] = gdf["area_gis"] * 0.01

In [6]:
def fix_known_issues(name):
    corrections = {
        "PerAo": "Peru",
        "Guyane FranAaise": "Guyane Francaise"
    }
    return corrections.get(name, name)

gdf["country"] = gdf["country"].apply(fix_known_issues)

In [7]:
gdf["country"].unique()


<StringArray>
[                       'Australia',                           'Brazil',
                         'Cambodia',                           'Canada',
                          'Bolivia',                           'Panama',
                      'New Zealand',                       'Costa Rica',
         'United States of America',              'Greenland (Denmark)',
                         'Malaysia',                     'South Africa',
                         'Botswana',                          'Ecuador',
         'United Kingdom - England',                          'Ireland',
                            'Chile',                           'Zambia',
              'Antigua and Barbuda',                         'Zimbabwe',
                           'Taiwan',                         'Paraguay',
                        'Nicaragua',                           'Guyana',
                         'Cameroon', 'Democratic Republic of the Congo',
                           'Mexico', 

In [8]:
gdf["country"].value_counts()

country
Mexico                              30644
New Zealand                         26592
United States of America            20104
United Kingdom - England             6982
India                                5636
Ireland                              4635
Spain                                3544
Italy                                3403
Canada                               3258
Chile                                3052
Taiwan                               2816
Peru                                 2530
Mozambique                           1882
Brazil                               1608
Australia                            1510
Colombia                             1244
Ecuador                               754
Nepal                                 739
Cameroon                              713
Paraguay                              678
Cambodia                              536
Bolivia                               321
Democratic Republic of the Congo      196
Namibia                   

In [9]:
area_by_country = (
    gdf.groupby("country")["area_gis"]
    .sum()
    .reset_index()
    .sort_values(by="area_gis", ascending=False)
)

In [10]:
area_by_country.head()

,country,area_gis
2,Australia,5.451420e+08
17,Greenland (Denmark),2.165116e+08
5,Brazil,1.625374e+08
27,Mexico,9.676475e+07
8,Canada,6.258754e+07


In [11]:
len(area_by_country)

48

In [12]:
gdf[["area_ofcl", "area_gis"]].describe()

,area_ofcl,area_gis
count,1.246160e+05,1.246160e+05
mean,7.424983e+03,1.221203e+04
std,1.620621e+05,6.400540e+05
min,0.000000e+00,4.646710e-10
25%,0.000000e+00,5.276108e+00
50%,0.000000e+00,6.559138e+01
75%,0.000000e+00,8.931469e+02
max,1.671251e+07,2.165116e+08


In [13]:
gdf["form_rec"].value_counts()


form_rec
Acknowledged by govt        122598
Not acknowledged by govt      2018
Name: count, dtype: int64

In [14]:
# Total territories
territories = gdf.groupby("country").size().reset_index(name="territories")

# Recognized (acknowledged)
recognized = (
    gdf[gdf["form_rec"] == "Acknowledged by govt"]
    .groupby("country")
    .size()
    .reset_index(name="recognized")
)

# Not recognized
not_recognized = (
    gdf[gdf["form_rec"] == "Not acknowledged by govt"]
    .groupby("country")
    .size()
    .reset_index(name="not_recognized")
)

# Area
area = (
    gdf.groupby("country")["area_km2"]
    .sum()
    .reset_index(name="area_km2")
)

# Merge everything
summary = territories.merge(recognized, on="country", how="left")
summary = summary.merge(not_recognized, on="country", how="left")
summary = summary.merge(area, on="country", how="left")

# Fill missing values
summary = summary.fillna(0)
summary["recognized"] = summary["recognized"].astype(int)
summary["not_recognized"] = summary["not_recognized"].astype(int)
summary["territories"] = summary["territories"].astype(int)

summary.head()

,country,territories,recognized,not_recognized,area_km2
0,Afghanistan,22,0,22,2.044225e+02
1,Antigua and Barbuda,1,1,0,1.512534e+02
2,Australia,1510,1407,103,5.451420e+06
3,Bolivia,321,280,41,2.850627e+05
4,Botswana,35,7,28,4.664814e+05


In [15]:
import pandas as pd
country_area = pd.DataFrame({
    "country": [
        "Australia", "Brazil", "Peru", "Bolivia", "Canada",
        "Mexico", "Colombia", "India", "Indonesia", "Kenya"
    ],
    "total_area": [
        7692024, 8515767, 1285216, 1098581, 9984670,
        1964375, 1141748, 3287263, 1904569, 580367
    ]
})

In [16]:
merged = summary.merge(country_area, on="country", how="left")

In [17]:
import pandas as pd

In [18]:
pd.crosstab(gdf["country"], gdf["form_rec"])


form_rec,Acknowledged by govt,Not acknowledged by govt
country,,
Afghanistan,0,22
Antigua and Barbuda,1,0
Australia,1407,103
Bolivia,280,41
Botswana,7,28
Brazil,1608,0
Cambodia,536,0
Cameroon,711,2
Canada,3258,0


In [19]:
crosstab = pd.crosstab(gdf["country"], gdf["form_rec"])

crosstab_percent = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

crosstab_percent.head()

form_rec,Acknowledged by govt,Not acknowledged by govt
country,,
Afghanistan,0.000000,100.000000
Antigua and Barbuda,100.000000,0.000000
Australia,93.178808,6.821192
Bolivia,87.227414,12.772586
Botswana,20.000000,80.000000


In [20]:
# Example: manually define a few countries (temporary)
country_area = pd.DataFrame({
    "country": ["Peru", "Brazil", "Australia"],
    "total_area": [1285216, 8515767, 7692024]  # km²
})

In [21]:
merged = summary.merge(country_area, on="country", how="left")

merged["indigenous_pct"] = (merged["area_km2"] / merged["total_area"]) * 100

merged[["country", "area_km2", "total_area", "indigenous_pct"]].head()

,country,area_km2,total_area,indigenous_pct
0,Afghanistan,2.044225e+02,NaN,NaN
1,Antigua and Barbuda,1.512534e+02,NaN,NaN
2,Australia,5.451420e+06,7692024.0,70.871068
3,Bolivia,2.850627e+05,NaN,NaN
4,Botswana,4.664814e+05,NaN,NaN


In [22]:
gdf["area_km2"] = gdf["area_gis"] * 0.01

In [23]:
area = (
    gdf.groupby("country")["area_km2"]
    .sum()
    .reset_index(name="area_km2")
)

In [24]:
merged.columns

Index(['country', 'territories', 'recognized', 'not_recognized', 'area_km2',
       'total_area', 'indigenous_pct'],
      dtype='str')

In [25]:
merged["indigenous_pct"] = (merged["area_km2"] / merged["total_area"]) * 100

In [26]:
merged.head()

,country,territories,recognized,not_recognized,area_km2,total_area,indigenous_pct
0,Afghanistan,22,0,22,2.044225e+02,NaN,NaN
1,Antigua and Barbuda,1,1,0,1.512534e+02,NaN,NaN
2,Australia,1510,1407,103,5.451420e+06,7692024.0,70.871068
3,Bolivia,321,280,41,2.850627e+05,NaN,NaN
4,Botswana,35,7,28,4.664814e+05,NaN,NaN


In [27]:
merged[["country", "territories", "area_km2", "indigenous_pct"]].sort_values(
    by="indigenous_pct", ascending=False
).head(10)

,country,territories,area_km2,indigenous_pct
2,Australia,1510,5.451420e+06,70.871068
5,Brazil,1608,1.625374e+06,19.086638
36,Peru,2530,2.145134e+05,16.690841
0,Afghanistan,22,2.044225e+02,NaN
1,Antigua and Barbuda,1,1.512534e+02,NaN
3,Bolivia,321,2.850627e+05,NaN
4,Botswana,35,4.664814e+05,NaN
6,Cambodia,536,1.503603e+04,NaN
7,Cameroon,713,4.319752e+04,NaN
8,Canada,3258,6.258754e+05,NaN


In [28]:
gdf[["area_gis", "area_km2"]].head()

,area_gis,area_km2
0,293432.545244,2934.325452
1,13491.658974,134.916590
2,642724.417132,6427.244171
3,47.442390,0.474424
4,59182.539128,591.825391


In [29]:
summary.sort_values(by="area_km2", ascending=False).head(10)

,country,territories,recognized,not_recognized,area_km2
2,Australia,1510,1407,103,5.451420e+06
17,Greenland (Denmark),1,0,1,2.165116e+06
5,Brazil,1608,1608,0,1.625374e+06
27,Mexico,30644,30644,0,9.676475e+05
8,Canada,3258,3258,0,6.258754e+05
46,Zambia,101,101,0,5.776456e+05
44,United States of America,20104,18891,1213,5.467995e+05
30,Namibia,158,158,0,4.809126e+05
4,Botswana,35,7,28,4.664814e+05
11,Colombia,1244,1244,0,4.117491e+05


In [30]:
summary.to_csv("../data/processed/landmark_country_summary.csv", index=False)